# 02 — Data Cleaning and Preparation

## Objective

This notebook will prepare the raw datasets collected in `01_data_collection.ipynb` for exploratory analysis and machine learning.

The datasets differ in their date ranges, reporting frequencies, column names and missing-value patterns. Before combining them, each dataset will be examined and cleaned individually so that they align appropriately.

The cleaning process will include:

- Converting date columns to datetime format.
- Removing non-data rows where necessary.
- Identifying and handling missing values.
- Standardising column names and data types.
- Aligning datasets with different reporting frequencies.
- Combining the datasets into a consistent time-series dataset.

Original files in `data/raw/` will remain unchanged and the clean outputs will be saved to `data/processed/`.

In [2]:
import pandas as pd

In [3]:
wti = pd.read_csv(
    "../data/raw/Cushing_OK_WTI_Spot_Price_FOB.csv",
    skiprows=4
)

wti.head()

,Day,Cushing OK WTI Spot Price FOB Dollars per Barrel
0,10/6/2026,96.24
1,10/5/2026,96.13
2,10/2/2026,97.89
3,10/1/2026,99.77
4,09/30/2026,97.18


In [4]:
wti.dtypes

Day                                                      str
Cushing OK WTI Spot Price FOB  Dollars per Barrel    float64
dtype: object

In [5]:
wti["Day"] = pd.to_datetime(wti["Day"], format="%m/%d/%Y")

In [6]:
wti.dtypes

Day                                                  datetime64[us]
Cushing OK WTI Spot Price FOB  Dollars per Barrel           float64
dtype: object

In [7]:
wti = wti.sort_values("Day")

In [8]:
wti.head()

,Day,Cushing OK WTI Spot Price FOB Dollars per Barrel
9516,1986-01-02,25.56
9515,1986-01-03,26.00
9514,1986-01-06,26.53
9513,1986-01-07,25.85
9512,1986-01-08,25.87


In [9]:
wti = wti.reset_index(drop=True)

In [10]:
wti.head()

,Day,Cushing OK WTI Spot Price FOB Dollars per Barrel
0,1986-01-02,25.56
1,1986-01-03,26.00
2,1986-01-06,26.53
3,1986-01-07,25.85
4,1986-01-08,25.87


In [11]:
wti.isnull().sum()

Day                                                  0
Cushing OK WTI Spot Price FOB  Dollars per Barrel    0
dtype: int64

In [12]:
wti["Day"].duplicated().sum()

np.int64(0)

### 2.1 WTI Crude-Oil Prices

The WTI dataset was loaded from the original EIA CSV file.

The `Day` column was converted from text to datetime and then sorted chronologically.

**Data quality checks:**
- Missing dates within existing rows: 0
- Missing oil-price values: 0
- Duplicate dates: 0

The original raw CSV file has been saved without modification as well.

In [18]:
wti.columns = ["Date", "WTI_Price"]

In [19]:
wti.head()

,Date,WTI_Price
0,1986-01-02,25.56
1,1986-01-03,26.00
2,1986-01-06,26.53
3,1986-01-07,25.85
4,1986-01-08,25.87


In [20]:
wti.to_csv("../data/processed/wti_clean.csv", index=False)

## 2.2 U.S. Commercial Crude-Oil Inventories

This section cleaned the weekly U.S. commercial crude-oil inventory data obtained from the EIA.

The original Excel file contains additional metadata rows, which will need to be removed before the dates and inventory values can be processed.

The cleaned dataset will retain the original weekly observations and will be saved separately from the raw data.

In [21]:
inventory = pd.read_excel("../data/raw/WCESTUS1w.xls", sheet_name="Data 1")

inventory.head()

,Back to Contents,Data 1: Weekly U.S. Ending Stocks excluding SPR of Crude Oil (Thousand Barrels)
0,Sourcekey,WCESTUS1
1,Date,Weekly U.S. Ending Stocks excluding SPR of Cru...
2,1982-08-20 00:00:00,338764
3,1982-08-27 00:00:00,336138
4,1982-09-24 00:00:00,335586


In [22]:
inventory = inventory.iloc[2:]

In [23]:
inventory.head()

,Back to Contents,Data 1: Weekly U.S. Ending Stocks excluding SPR of Crude Oil (Thousand Barrels)
2,1982-08-20 00:00:00,338764
3,1982-08-27 00:00:00,336138
4,1982-09-24 00:00:00,335586
5,1982-10-01 00:00:00,334786
6,1982-10-08 00:00:00,335260


In [24]:
inventory.columns = ["Date", "Crude_Inventory"]

In [25]:
inventory.head()

,Date,Crude_Inventory
2,1982-08-20 00:00:00,338764
3,1982-08-27 00:00:00,336138
4,1982-09-24 00:00:00,335586
5,1982-10-01 00:00:00,334786
6,1982-10-08 00:00:00,335260


In [26]:
inventory.dtypes

Date               object
Crude_Inventory    object
dtype: object

In [27]:
inventory["Date"] = pd.to_datetime(inventory["Date"])

In [28]:
inventory["Crude_Inventory"] = pd.to_numeric(inventory["Crude_Inventory"])

In [29]:
inventory.dtypes

Date               datetime64[us]
Crude_Inventory             int64
dtype: object

In [30]:
inventory = inventory.reset_index(drop=True)

In [31]:
inventory.head()

,Date,Crude_Inventory
0,1982-08-20,338764
1,1982-08-27,336138
2,1982-09-24,335586
3,1982-10-01,334786
4,1982-10-08,335260


In [32]:
inventory.isnull().sum()

Date               0
Crude_Inventory    0
dtype: int64

In [33]:
inventory["Date"].duplicated().sum()

np.int64(0)

### Initial Cleaning Results

The first two rows of the original Excel worksheet contained metadata rather than inventory observations so I removed them.

The columns were renamed to `Date` and `Crude_Inventory`. The date column was converted to datetime format, while inventory values were converted to integers. The index was also reset.

**Data quality checks:**
- Missing dates: 0
- Missing inventory values: 0
- Duplicate dates: 0

The inventory values remain in their original units of **thousand barrels**.

The cleaned dataset retains its weekly reporting frequency. Differences between weekly inventory observations and daily WTI prices will be addressed when the datasets are combined.

In [34]:
inventory.to_csv("../data/processed/inventory_clean.csv", index=False)